<a href="https://colab.research.google.com/github/thakur785/sandbox_llm/blob/ai-agents/AIAgents/AIAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#learning AI agents using langgraph

In [8]:
from dotenv import load_dotenv
_ = load_dotenv()

In [9]:
!pip install -q langchain langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.0 MB/s eta 0:00:00


In [10]:
!pip install -q langchain_community

In [11]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

/tmp/ipykernel_559/2233616850.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [13]:
tool = TavilySearchResults(max_results=4) #increased number of results
print(type(tool))
print(tool.name)

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

# AI Agent in langgraph
## Agentic search tool:


In [3]:
!pip install tavily-python

In [4]:
#libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient
# load environment variable from .env file
_ = load_dotenv()


In [5]:
#connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [6]:
#  run search
result = client.search("what is in Nvidia's new Blackwell GPU?", include_answer=True)
#print the answer
result["answer"]

"The Blackwell GPU features 5th Gen Tensor Cores and 4th Gen RT cores for AI and graphics. It supports FP4 precision and doubles ray tracing performance. It's designed for large-scale AI workloads."

In [7]:
#weather serach
#query
city = "Bengaluru"
query = f"""
        what is the traffic in {city} in 2026?
"""

In [8]:
# run the search
result = client.search(query, max_results=1)
#print first result

data = result["results"][0]["content"]
print(data)

Never miss a post from supri\_v. Sign up for Instagram to stay in the loop. Bengaluru records the highest traffic of 2026. hopr.mobi's profile picture. @supri\_v highest traffic of 2026 built one solo car at a time 😤 if every corporate employee on this stretch shared a ride, 70% of this jam doesn't exist tomorrow 🚗. ORR is filled with traffic all the days 😟. Why all the company are at the same place. This is what netizens are saying, whole world is in banglore now since years. sudhash.r's profile picture. Please start a rules if a car having one person will not be allowed in High traffic areas. anamul.king_20.07's profile picture. abhisek.bedant's profile picture. Today's update yet to come wait for the HAL traffic update 🔥🔥🔥🔥. Let's see the magic it does. It is instagram's algorithm or some magic. What are your thoughts on the CBSE OSM controversy? RCB is not just a team… it’s an emotion.


## Persistence and streaming


In [14]:
from dotenv import load_dotenv
_ = load_dotenv()

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [16]:
from google.colab import userdata
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [17]:
tool = TavilySearchResults(max_results=2)

In [18]:
class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]

In [19]:
!pip install -q langgraph.checkpoint.sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 5.0 MB/s eta 0:00:00


In [26]:
!pip install -q langchain-groq

In [27]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()

In [65]:
# create agent with persistency
class Agent:
  def __init__(self, model, tools, checkpointer, system=""):
    self.system = system
    graph = StateGraph(AgentState)
    graph.add_node("llm", self.call_openai)
    graph.add_node("action", self.take_action)
    graph.add_conditional_edges("llm", self.exists_action, {True: "action", False:END})
    graph.add_edge("action", "llm")
    graph.set_entry_point("llm")
    self.graph = graph.compile(checkpointer=checkpointer)
    self.tools = {t.name: t for t in tools}
    self.model = model.bind_tools(tools)

  def call_openai(self, state: AgentState):
    messages = state["messages"]
    if self.system:
      messages = [SystemMessage(content=self.system)] + messages
    message = self.model.invoke(messages)
    return {'messages': [message]}

  def exists_action(self, state: AgentState):
    result = state["messages"][-1]
    return len(result.tool_calls) > 0

  def take_action(self, state: AgentState):
    tool_calls = state["messages"][-1].tool_calls
    results = []
    for t in tool_calls:
      print (f"calling: {t}")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id=t['name'], content=str(result)))
      print("Back to Model!")
      return {'messages': results}

In [66]:
print(type(memory))


<class 'langgraph.checkpoint.memory.InMemorySaver'>


In [67]:
from google.colab import userdata
from langchain_groq import ChatGroq
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatGroq(model="llama-3.3-70b-versatile")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [68]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [69]:
thread = {"configurable": {"thread_id": "1"}}

In [70]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4qwcmetjh', 'function': {'arguments': '{"query":"San Francisco current weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 789, 'total_tokens': 810, 'completion_time': 0.028614403, 'completion_tokens_details': None, 'prompt_time': 0.038988154, 'prompt_tokens_details': None, 'queue_time': 0.008942405, 'total_time': 0.067602557}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea2a4-f71f-70b3-ad8f-a91a13196b76-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco current weather'}, 'id': '4qwcmetjh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 789, 'output_tokens': 21, 'total_tokens': 810})]
calling: {'name': '

In [71]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6kq7k5r99', 'function': {'arguments': '{"query":"Los Angeles current weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 2001, 'total_tokens': 2022, 'completion_time': 0.021682385, 'completion_tokens_details': None, 'prompt_time': 0.302074527, 'prompt_tokens_details': None, 'queue_time': 0.023049018, 'total_time': 0.323756912}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea2a5-9e93-71e3-825b-29ff004f25dc-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Los Angeles current weather'}, 'id': '6kq7k5r99', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2001, 'output_tokens': 21, 'total_tokens': 2022})]}
calli

In [72]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0vpx0bv88', 'function': {'arguments': '{"query":"San Francisco vs Los Angeles temperature comparison"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 3287, 'total_tokens': 3311, 'completion_time': 0.060431522, 'completion_tokens_details': None, 'prompt_time': 0.218188102, 'prompt_tokens_details': None, 'queue_time': 0.009160509, 'total_time': 0.278619624}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea2a5-d602-7fa2-8810-ebbd512b3e19-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco vs Los Angeles temperature comparison'}, 'id': '0vpx0bv88', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3287, 'ou